### Intuition on EMD: Moving sand from a heap to another: loss function=(weight of sand)*(distance)


### f<i,j> represents the mass trnasported from i to j

obj_function:$$\min_{f_{i,j}} \sum_{i=1}^{m} \sum_{j=1}^{n} f_{i,j} \cdot d_{i,j}$$

restriction:
\begin{aligned}
& \sum_{j=1}^{n} f_{i,j} \leq w_{pi}, \quad i=1, \dots, m \\
& \sum_{i=1}^{m} f_{i,j} \leq w_{qj}, \quad j=1, \dots, n \\
& \sum_{i=1}^{m} \sum_{j=1}^{n} f_{i,j} = \min \left( \sum_{i=1}^{m} w_{pi}, \sum_{j=1}^{n} w_{qj} \right) \\
& f_{i,j} \geq 0, \quad i=1, \dots, m, \quad j=1, \dots, n
\end{aligned}
#### third formula:The constraint ensures that the total flow equals the minimum of the total weights of the two distributions(supply and demand)

$$\text{EMD}(P, Q) = \min_{F} \sum_{i,j} f_{ij} \cdot d_{ij}$$

linear programming:
$$\min_{\mathbf{f}}\ \mathbf{c}^\top \mathbf{f} \quad \text{s.t.}\ A\mathbf{f} = \mathbf{b},\ \mathbf{f} \geq 0$$

def of distance:
$$c_{ij} = 1 - \frac{u_i^\top v_j}{\|u_i\|\,\|v_j\|}$$

dot product of whole set of p with the average of j set
$$s_i = \max\!\left\{u_i^\top \cdot \frac{\sum_j v_j}{HW},\ 0\right\}$$

$$\hat{s}_i = \frac{s_i \cdot HW}{\sum_j s_j}$$

$$F^* = \text{diag}(u) \cdot K \cdot \text{diag}(v)$$

#### mar12th/daily note: get a rough understanding of mathematical part of this essay

#### to-do: implement a simple replication in python(using numpy only)

In [3]:
import numpy as np
from scipy.optimize import linprog
k=4
seed=42
np.random.seed(seed)
p=np.random.rand(k, k)
q=np.random.rand(k, k)
print(p,q)
q_average=np.mean(q,axis=0)
p_average=np.mean(p,axis=0)
print(q_average)

[[0.37454012 0.95071431 0.73199394 0.59865848]
 [0.15601864 0.15599452 0.05808361 0.86617615]
 [0.60111501 0.70807258 0.02058449 0.96990985]
 [0.83244264 0.21233911 0.18182497 0.18340451]] [[0.30424224 0.52475643 0.43194502 0.29122914]
 [0.61185289 0.13949386 0.29214465 0.36636184]
 [0.45606998 0.78517596 0.19967378 0.51423444]
 [0.59241457 0.04645041 0.60754485 0.17052412]]
[0.49114492 0.37396917 0.38282708 0.33558739]


In [4]:
def compute_weight(p,q_average):
    res=np.zeros((p.shape[1]))
    sum=0
    for i in range(p.shape[1]):
        ins=p[:,i].dot(q_average)
        res[i]=ins
        sum+=ins
    #Marginal normalization
    return res/sum
##cross-reference: the formula in the first cell, which is the dot product of whole set of p with the average of j set
p_new=compute_weight(p,q_average)
q_new=compute_weight(q,p_average)
print(p_new)
print(q_new)

[0.24092998 0.27804897 0.144259   0.33676205]
[0.33240516 0.19160534 0.27943889 0.19655061]


In [5]:
def compute_cos(p,q):
    p_norm=np.linalg.norm(p,axis=0)
    q_norm=np.linalg.norm(q,axis=0)
    cos=np.zeros((p.shape[1],p.shape[1]))
    for i in range(p.shape[1]):
        for j in range(q.shape[1]):
            cos[i,j]=p[:,i].dot(q[:,j])/(p_norm[i]*q_norm[j])
    return cos
cos_mat=compute_cos(p,q)
sin_mat=np.sqrt(1-cos_mat**2)
distance_mat=(np.linalg.norm(p,axis=0))*sin_mat
print(distance_mat)

[[0.53760728 0.8780146  0.30617194 0.90139638]
 [0.8119571  0.42710189 0.52110403 0.7725452 ]
 [0.96260995 0.9924078  0.52751452 1.24186841]
 [0.55575961 0.59098624 0.55076692 0.21524487]]


In [ ]:
print(f"distance_mat shape: {distance_mat.shape}")   
print(f"c shape: {distance_mat.flatten().shape}")     


distance_mat shape: (4, 4)
c shape: (16,)


In [ ]:
print(np.any(np.isnan(distance_mat))) 
print(np.any(np.isinf(distance_mat)))  


False
False


In [ ]:
b_eq=np.concatenate([p_new,q_new])

In [20]:
m, n = 4, 4
A_row = np.kron(np.eye(4), np.ones((1, 4)))   # (4, 16)
A_col = np.kron(np.ones((1, 4)), np.eye(4))   # (4, 16)
A_eq  = np.vstack([A_row, A_col])             # (8, 16)

In [22]:
result = linprog(
    distance_mat.flatten(),           
    A_eq=A_eq, 
    b_eq=b_eq,
    bounds=[(0,None)] * 16, 
    method='highs'
) 

In [23]:
print(result.status)

0


---
## Comparison: Group A vs Group B
- **Group A**: Two independent random matrices `p` and `q`
- **Group B**: Matrix `p` vs its right-shifted version (last column wraps to first position)

In [ ]:
def compute_emd(mat_a, mat_b):
    """Compute EMD between two matrices using DeepEMD formulation."""
    k = mat_a.shape[1]
    b_avg = np.mean(mat_b, axis=0)
    a_avg = np.mean(mat_a, axis=0)
    w_a = compute_weight(mat_a, b_avg)
    w_b = compute_weight(mat_b, a_avg)

    cos = compute_cos(mat_a, mat_b)
    sin = np.sqrt(np.clip(1 - cos**2, 0, None))
    dist = np.linalg.norm(mat_a, axis=0) * sin

    b_eq = np.concatenate([w_a, w_b])
    A_row = np.kron(np.eye(k), np.ones((1, k)))
    A_col = np.kron(np.ones((1, k)), np.eye(k))
    A_eq = np.vstack([A_row, A_col])

    res = linprog(
        dist.flatten(),
        A_eq=A_eq, b_eq=b_eq,
        bounds=[(0, None)] * (k * k),
        method='highs'
    )
    return res.fun, res.status

### Group A: Two independent random matrices (`p` vs `q`)

In [ ]:
emd_A, status_A = compute_emd(p, q)
print(f"Group A (random p vs random q)")
print(f"  EMD = {emd_A:.6f}  |  solver status = {status_A}")

### Group B: `p` vs right-shifted `p`
Shift all columns of `p` one position to the right; the displaced last column wraps back to the first position (`np.roll(p, 1, axis=1)`).

In [ ]:
p_shifted = np.roll(p, 1, axis=1)   # last column -> first column
print("p (original):")
print(p)
print("p_shifted (right-shift by 1 col, last col wraps to first):")
print(p_shifted)

In [ ]:
emd_B, status_B = compute_emd(p, p_shifted)
print(f"Group B (p vs right-shifted p)")
print(f"  EMD = {emd_B:.6f}  |  solver status = {status_B}")

### Summary
Group B should produce a **lower** EMD than Group A, because the shifted matrix is structurally
similar to the original (same values, only column order changed), whereas Group A compares
two unrelated random matrices.

In [ ]:
print("=== EMD Comparison ===")
print(f"Group A | p vs q (independent random)  : EMD = {emd_A:.6f}")
print(f"Group B | p vs p_shifted (1-col shift)  : EMD = {emd_B:.6f}")
print(f"\nGroup B / Group A ratio: {emd_B/emd_A:.4f}")
print("(ratio < 1 => shifted pair is more similar, as expected)")